# Aşama 3 — Structured Streaming ile Gerçek Zamanlı Illicit İşlem İzleme

```
 producer.py ──JSON──▶  Kafka topic "btc-tx"  ──▶  Spark Structured Streaming
 (test dönemi,          (3 partition)              │
  adım 35-49,                                      ├─ from_json + çevrimiçi özellik hesabı (fee_rate, log)
  %2 geç kayıt)                                    ├─ stream-static JOIN ◀── feature store (Parquet, Aşama 1)
                                                   ├─ RF PipelineModel.transform ◀── Aşama 2'de eğitilen model
                                                   │
                                                   ├─▶ Sorgu A: alarm akışı (p_illicit ≥ 0.5)  → Parquet file sink (exactly-once)
                                                   ├─▶ Sorgu B: 30 sn pencereli istatistik, 1 dk watermark → foreachBatch (update)
                                                   └─▶ Sorgu C: canlı model izleme (gerçek etiketle) → foreachBatch
```

**Neden bu tasarım?**
* Mesaj yalnızca işlemin zincirde hemen görünen alanlarını taşıyor (boyut, ücret, girdi/çıktı sayısı, BTC). Graf ve komşuluk tabanlı 165 anonim özellik ise önceden hesaplanmış bir **feature store**'dan join ile geliyor. Gerçek fraud sistemleri de online ve offline özellikleri bu şekilde birleştirir.
* Model, Aşama 2'de kaydedilen `PipelineModel`. Imputer ve VectorAssembler dahil aynı dönüşümler streaming DataFrame'e de uygulanıyor; eğitim ile servis arasında kod farkı (training/serving skew) oluşmuyor.
* Producer, modelin **hiç görmediği** test adımlarını (35–49) oynatıyor.

**Canlı demo:** Önce bu notebook'u §4'e kadar çalıştırın, ardından terminalde producer'ı başlatın:
```bash
docker compose exec jupyter python streaming/producer.py --rate 300 --step-pause 3
```

In [1]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(ROOT))
from common.spark_utils import *  # get_spark, RAW, PROC, MODELS, CHECKPOINTS, TRAIN_MAX_STEP ...
import pandas as pd
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)

In [2]:
import os, json, time, shutil, threading, subprocess
from pyspark.sql import functions as F
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
from IPython.display import display, clear_output

spark = get_spark("03-structured-streaming", {"spark.sql.shuffle.partitions": "4"})
DEMO_SECONDS = int(os.environ.get("DEMO_SECONDS", "120"))
EMBEDDED_PRODUCER = os.environ.get("DEMO_EMBEDDED_PRODUCER", "0") == "1"
CK = CHECKPOINTS / "streaming"
ALERTS_OUT = ROOT / "data" / "stream_out" / "alerts"
shutil.rmtree(CK, ignore_errors=True); shutil.rmtree(ALERTS_OUT, ignore_errors=True)   # temiz demo
print("Kafka:", KAFKA_BOOTSTRAP, "| topic:", KAFKA_TOPIC, "| demo süresi:", DEMO_SECONDS, "sn")

:: loading settings :: url = jar:file:/usr/local/lib/python3.13/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Kafka: localhost:9092 | topic: btc-tx | demo süresi: 180 sn


## 1. Kafka topic'ini sıfırla (her demo temiz başlasın)

In [3]:
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import UnknownTopicOrPartitionError, TopicAlreadyExistsError

admin = KafkaAdminClient(bootstrap_servers=KAFKA_BOOTSTRAP)
try:
    admin.delete_topics([KAFKA_TOPIC]); time.sleep(3)
except UnknownTopicOrPartitionError:
    pass
for _ in range(10):
    try:
        admin.create_topics([NewTopic(KAFKA_TOPIC, num_partitions=3, replication_factor=1)]); break
    except TopicAlreadyExistsError:
        time.sleep(1)
print("topic'ler:", [t for t in admin.list_topics() if not t.startswith("__")]); admin.close()

topic'ler: ['btc-tx']


## 2. Model, feature store ve etiket tablosu (statik taraf)

In [4]:
model = PipelineModel.load(str(MODELS / "rf_illicit"))
meta = json.loads((MODELS / "rf_illicit_meta.json").read_text())
FEATURES, THRESHOLD = meta["features"], meta["threshold"]

ONLINE = ["size", "num_input_addresses", "num_output_addresses", "fee_rate_sat_b", "log_fees", "log_out_BTC_total"]
txs = spark.read.parquet(str(PROC / "txs"))
feature_store = txs.select("txId", *[c for c in FEATURES if c not in ONLINE]).cache()
labels = txs.select("txId", "label", "class_name").cache()
print(f"feature store: {feature_store.count()} satır × {len(feature_store.columns) - 1} özellik | online özellik: {len(ONLINE)}")

feature store: 203769 satır × 177 özellik | online özellik: 6


## 3. Streaming pipeline

`readStream` aslında henüz hiçbir şey okumaz. Kurduğumuz şey **sınırsız (unbounded) bir tablonun mantıksal planı**; Spark bu planı her micro-batch'te yalnızca yeni gelen kayıtlar üzerinde artımlı (incremental) olarak çalıştırır.

In [5]:
raw = (spark.readStream.format("kafka")
       .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
       .option("subscribe", KAFKA_TOPIC)
       .option("startingOffsets", "earliest")
       .option("maxOffsetsPerTrigger", 4000)          # backpressure: batch başına üst sınır
       .option("failOnDataLoss", "false")
       .load())
print("isStreaming:", raw.isStreaming); raw.printSchema()

isStreaming: True
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [6]:
EVENT_SCHEMA = ("txId LONG, time_step INT, size DOUBLE, fees DOUBLE, num_input_addresses DOUBLE, "
                "num_output_addresses DOUBLE, out_BTC_total DOUBLE, event_time TIMESTAMP")

events = (raw.select(F.from_json(F.col("value").cast("string"), EVENT_SCHEMA,
                                 {"timestampFormat": "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"}).alias("e"),
                     F.col("timestamp").alias("kafka_ts"), "partition", "offset")
             .select("e.*", "kafka_ts", "partition", "offset")
             # çevrimiçi özellikler: Aşama 1'deki dönüşümlerin aynısı
             .withColumn("fee_rate_sat_b", F.col("fees") * 1e8 / F.col("size"))
             .withColumn("log_fees", F.log1p(F.col("fees") * 1e8))
             .withColumn("log_out_BTC_total", F.log1p(F.col("out_BTC_total") * 1e8)))

enriched = events.join(feature_store, "txId", "left")               # stream-static join
scored = (model.transform(enriched)
          .withColumn("p_illicit", F.round(vector_to_array("probability")[1], 4))
          .withColumn("flag", (F.col("p_illicit") >= THRESHOLD).cast("int"))
          .select("txId", "time_step", "event_time", "kafka_ts", "partition", "offset",
                  "out_BTC_total", "fee_rate_sat_b", "p_illicit", "flag"))
scored.printSchema()

root
 |-- txId: long (nullable = true)
 |-- time_step: integer (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- kafka_ts: timestamp (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- out_BTC_total: double (nullable = true)
 |-- fee_rate_sat_b: double (nullable = true)
 |-- p_illicit: double (nullable = true)
 |-- flag: integer (nullable = true)



### Sorgu A — Alarm akışı
`append` modu: yalnızca eşiği geçen yeni kayıtlar sink'e eklenir; durum (state) tutulmaz. Sink olarak **Parquet file sink** kullanıyoruz. File sink, her batch'in yazdığı dosyaları `_spark_metadata` günlüğüne commit eder; yarıda kalmış bir batch'in dosyaları okuyucuya görünmez. Tekrar oynatılabilir bir kaynakla (Kafka) birlikte bu düzen **uçtan uca exactly-once** sağlar (§5'te doğruluyoruz).

> Not: Her sorgu Kafka'yı bağımsız okur ve join ile modeli ayrı çalıştırır. Üç sorgu, işin üç kez yapılması demek. Üretimde tek sorgu + `foreachBatch` ile çıktıları dağıtmak daha verimli olurdu; burada her sink türünün semantiğini ayrı ayrı göstermek için ayrık bıraktık.

In [7]:
def start_alerts(name="alerts"):
    return (scored.filter("flag = 1")
            .writeStream.queryName(name).format("parquet").outputMode("append")
            .option("path", str(ALERTS_OUT)).option("checkpointLocation", str(CK / "alerts"))
            .trigger(processingTime="3 seconds").start())

q_alerts = start_alerts()

### Sorgu B — Pencereli istatistik + watermark
* `window(event_time, "30 seconds")`: **olay zamanına** göre 30 saniyelik pencereler. Kayıt Kafka'ya ne zaman ulaşmış olursa olsun, ait olduğu pencereye sayılır.
* `withWatermark("event_time", "1 minute")`: Spark, görülen en büyük olay zamanının 1 dakika gerisini watermark olarak tutar. Bundan daha eski bir pencereye ait geç kayıt **atılır** ve o pencerenin state'i bellekten silinir. Watermark olmasaydı state sonsuza kadar büyürdü.
* Producer kayıtların %2'sini 150 saniye geç damgalıyor. Bu kayıtların atıldığını `numRowsDroppedByWatermark` metriğinden göreceğiz.
* `update` modu: her batch'te yalnızca değişen pencereler yayımlanır. `foreachBatch` ile bunları sürücüdeki bir tabloya yazıyoruz.

In [8]:
window_state, wlock = {}, threading.Lock()

def upsert_windows(batch_df, batch_id):
    rows = batch_df.collect()
    with wlock:
        for r in rows:
            window_state[r["window"]["start"]] = {**r.asDict(), "window": f'{r["window"]["start"]:%H:%M:%S}-{r["window"]["end"]:%H:%M:%S}', "batch": batch_id}

window_stats = (scored.withWatermark("event_time", "1 minute")
                .groupBy(F.window("event_time", "30 seconds"))
                .agg(F.count("*").alias("işlem"), F.sum("flag").alias("alarm"),
                     F.round(F.avg("p_illicit"), 4).alias("ort_p"), F.round(F.sum("out_BTC_total"), 2).alias("BTC_hacmi"),
                     F.min("time_step").alias("adım_min"), F.max("time_step").alias("adım_max")))

q_windows = (window_stats.writeStream.queryName("windows").outputMode("update")
             .foreachBatch(upsert_windows).option("checkpointLocation", str(CK / "windows"))
             .trigger(processingTime="5 seconds").start())

### Sorgu C — Canlı model izleme
Gerçek sistemde etiket gecikmeli gelir; burada veri setindeki etiketler "sonradan gelen analist kararı" yerine geçiyor. Her micro-batch etiket tablosuyla birleştirilip kümülatif karmaşıklık matrisi güncelleniyor.

In [9]:
monitor = {"TP": 0, "FP": 0, "FN": 0, "TN": 0, "unknown_flagged": 0, "unknown_total": 0, "batches": 0}
per_step = {}                                   # zaman adımı -> {"TP":..,"FP":..,"FN":..,"TN":..}
mlock = threading.Lock()

def update_monitor(batch_df, batch_id):
    rows = (batch_df.join(labels, "txId", "left")
            .groupBy("time_step", "class_name", "flag").count().collect())
    with mlock:
        monitor["batches"] += 1
        for r in rows:
            c, f, n = r["class_name"], r["flag"], r["count"]
            step = per_step.setdefault(r["time_step"], {"TP": 0, "FP": 0, "FN": 0, "TN": 0})
            if c in ("illicit", "licit"):
                key = ("TP" if f else "FN") if c == "illicit" else ("FP" if f else "TN")
                monitor[key] += n; step[key] += n
            else:
                monitor["unknown_total"] += n; monitor["unknown_flagged"] += n if f else 0

q_monitor = (scored.writeStream.queryName("monitor").foreachBatch(update_monitor)
             .option("checkpointLocation", str(CK / "monitor")).trigger(processingTime="5 seconds").start())
print([q.name for q in spark.streams.active])

['alerts', 'windows', 'monitor']


## 4. Canlı pano

Producer'ı **şimdi** başlatın (ya da `DEMO_EMBEDDED_PRODUCER=1` ile notebook başlatsın). Pano `DEMO_SECONDS` boyunca 5 saniyede bir yenilenir.

Sunumda zaman kısıtlıysa `--from-step 40` ile başlatın. Böylece 43. adımdaki kırılma birkaç dakika içinde canlı izleme tablosunda görünür.

In [10]:
producer_proc = None
if EMBEDDED_PRODUCER:
    producer_proc = subprocess.Popen(
        [sys.executable, str(ROOT / "streaming" / "producer.py"), "--rate", os.environ.get("PRODUCER_RATE", "300"),
         "--from-step", os.environ.get("PRODUCER_FROM_STEP", "35"),
         "--step-pause", "2", "--bootstrap", KAFKA_BOOTSTRAP, "--topic", KAFKA_TOPIC],
        stdout=open(CHECKPOINTS / "producer.log", "w"), stderr=subprocess.STDOUT)
    print("gömülü producer başlatıldı, pid", producer_proc.pid)

gömülü producer başlatıldı, pid 32117


In [11]:
def metrics():
    with mlock:
        m = dict(monitor)
    p = m["TP"] / (m["TP"] + m["FP"]) if m["TP"] + m["FP"] else 0
    r = m["TP"] / (m["TP"] + m["FN"]) if m["TP"] + m["FN"] else 0
    return {**m, "precision": round(p, 3), "recall": round(r, 3), "F1": round(2 * p * r / (p + r), 3) if p + r else 0}

def dropped_total(q):
    return sum(((p.get("stateOperators") or [{}])[0].get("numRowsDroppedByWatermark") or 0) for p in q.recentProgress)

def step_table():
    with mlock:
        rows = [{"adım": s, **c} for s, c in sorted(per_step.items())]
    df = pd.DataFrame(rows)
    if df.empty: return df
    df["illicit"] = df.TP + df.FN
    df["F1"] = (2 * df.TP / (2 * df.TP + df.FP + df.FN)).where(df.illicit > 0).round(3)
    return df.set_index("adım")[["illicit", "TP", "FN", "FP", "F1"]].T

def read_alerts():
    try:
        return spark.read.parquet(str(ALERTS_OUT))
    except Exception:
        return None

def progress_row(q):
    lp = q.lastProgress or {}
    st = (lp.get("stateOperators") or [{}])[0]
    return {"sorgu": q.name, "batch": lp.get("batchId"), "girdi satır": lp.get("numInputRows"),
            "girdi/sn": round(lp.get("inputRowsPerSecond") or 0, 1), "işlenen/sn": round(lp.get("processedRowsPerSecond") or 0, 1),
            "süre ms": (lp.get("durationMs") or {}).get("triggerExecution"),
            "state satır": st.get("numRowsTotal"), "watermark ile atılan (toplam)": dropped_total(q) if st else None,
            "watermark": (lp.get("eventTime") or {}).get("watermark")}

def dashboard():
    clear_output(wait=True)
    print(f"⏱  {time.strftime('%H:%M:%S')}   aktif sorgular: {[q.name for q in spark.streams.active]}")
    display(pd.DataFrame([progress_row(q) for q in (q_alerts, q_windows, q_monitor)]))
    print("— Son alarmlar (p_illicit ≥ eşik) —")
    a = read_alerts()
    if a is not None:
        print(f"   toplam alarm dosyada: {a.count()}")
        display(a.select("time_step", "txId", "p_illicit", "out_BTC_total", F.round("fee_rate_sat_b", 1).alias("fee_rate"),
                         "partition", "offset").orderBy(F.desc("kafka_ts")).limit(8).toPandas())
    print("— 30 sn pencereler (olay zamanı) —")
    with wlock:
        w = pd.DataFrame(sorted(window_state.values(), key=lambda r: r["window"])[-6:])
    display(w)
    print("— Canlı model izleme (kümülatif) —")
    display(pd.DataFrame([metrics()]))
    print("— Canlı model izleme (zaman adımı bazında) —")
    display(step_table())

t_end = time.time() + DEMO_SECONDS
while time.time() < t_end:
    dashboard(); time.sleep(5)
dashboard()

⏱  15:47:16   aktif sorgular: ['alerts', 'windows', 'monitor']


,sorgu,batch,girdi satır,girdi/sn,işlenen/sn,süre ms,state satır,watermark ile atılan (toplam),watermark
0,alerts,24,624,124.7,100.9,6182,NaN,NaN,NaN
1,windows,16,971,100.2,131.2,7401,4.0,19.0,2026-11-24T12:45:57.854Z
2,monitor,15,1926,236.6,190.2,10127,NaN,NaN,NaN


— Son alarmlar (p_illicit ≥ eşik) —


   toplam alarm dosyada: 3182


,time_step,txId,p_illicit,out_BTC_total,fee_rate,partition,offset
0,49,158571082,0.9700,0.594919,202.7,1,15442
1,49,157585130,0.5500,0.078261,38.8,1,15429
2,49,136707342,0.5582,0.023776,99.0,0,15563
3,49,188612885,0.9900,0.253634,241.3,0,15559
4,49,132185328,0.5627,0.373623,122.5,2,15588
5,49,175742370,0.6700,0.000620,149.2,0,15546
6,49,194312038,0.5500,0.021777,153.8,0,15535
7,49,157622101,0.8697,0.002196,123.1,2,15565


— 30 sn pencereler (olay zamanı) —


,window,işlem,alarm,ort_p,BTC_hacmi,adım_min,adım_max,batch
0,15:44:30-15:45:00,8072,723,0.1932,138272.15,40,42,4
1,15:45:00-15:45:30,8673,595,0.1601,143890.75,42,43,6
2,15:45:30-15:46:00,8168,503,0.1565,182166.43,43,45,9
3,15:46:00-15:46:30,8278,521,0.1606,172715.57,45,47,12
4,15:46:30-15:47:00,8285,463,0.1706,112583.30,47,49,16
5,15:47:00-15:47:30,289,14,0.1717,2617.73,49,49,16


— Canlı model izleme (kümülatif) —


,TP,FP,FN,TN,unknown_flagged,unknown_total,batches,precision,recall,F1
0,371,13,265,10535,2798,35463,16,0.966,0.583,0.727


— Canlı model izleme (zaman adımı bazında) —


adım,40,41,42,43,44,45,46,47,48,49
illicit,112.000,116.00,239.000,24.0,24.000,5.0,2.0,22.0,36.0,56.000
TP,72.000,107.00,189.000,0.0,1.000,0.0,1.0,0.0,0.0,1.000
FN,40.000,9.00,50.000,24.0,23.000,5.0,1.0,22.0,36.0,55.000
FP,1.000,0.00,4.000,0.0,5.000,1.0,1.0,1.0,0.0,0.000
F1,0.778,0.96,0.875,0.0,0.067,0.0,0.5,0.0,0.0,0.035


## 5. Hata toleransı: checkpoint'ten kaldığı yerden devam

Her sorgu checkpoint dizinine iki günlük yazar: `offsets/N` (batch N için **planlanan** Kafka offset aralığı, batch başlamadan önce yazılır, write-ahead log) ve `commits/N` (batch N sink'e başarıyla yazıldı). Sorgu bir batch'in ortasında ölürse, yeniden başlatıldığında `offsets` günlüğünde olup `commits` günlüğünde olmayan batch **aynı offset aralığıyla** yeniden çalıştırılır. File sink, o batch'in daha önce yarım kalmış dosyalarını `_spark_metadata` sayesinde yok sayar.

Aşağıda sorguyu bir batch'in **ortasında** durdurup aynı checkpoint ile yeniden başlatıyoruz, sonra çıktıda tekrar eden kayıt olup olmadığını kontrol ediyoruz.

In [12]:
def last_id(d):
    ids = [int(p.name) for p in d.iterdir() if p.name.isdigit()]
    return max(ids) if ids else None

for _ in range(50):                                   # bir batch çalışırken yakala
    if q_alerts.status["isTriggerActive"]: break
    time.sleep(0.2)
q_alerts.stop()
off_id, com_id = last_id(CK / "alerts" / "offsets"), last_id(CK / "alerts" / "commits")
planned = json.loads((CK / "alerts" / "offsets" / str(off_id)).read_text().splitlines()[-1])
print(f"son planlanan batch: {off_id} | son commit edilen batch: {com_id}")
print(f"batch {off_id} offset aralığı sonu: {planned}")
print("→", "batch yarıda kesildi, yeniden başlatmada TEKRAR çalıştırılacak" if off_id != com_id else "kesinti batch arasına denk geldi, sıradaki batch yeni offset'lerden başlayacak")
time.sleep(4)

q_alerts = start_alerts("alerts_resumed")
for _ in range(60):
    time.sleep(1)
    if q_alerts.recentProgress: break
first = q_alerts.recentProgress[0]
print(f"\nyeniden başlatılan sorgunun ilk batch'i: {first['batchId']} | endOffset: {first['sources'][0]['endOffset']}")
time.sleep(8)
a = read_alerts()
n, nd = a.count(), a.select("txId").distinct().count()
print(f"alarm dosyası: {n} satır, {nd} tekil txId → tekrar eden kayıt: {n - nd}")

son planlanan batch: 24 | son commit edilen batch: 24
batch 24 offset aralığı sonu: {'btc-tx': {'2': 15620, '1': 15446, '0': 15581}}
→ kesinti batch arasına denk geldi, sıradaki batch yeni offset'lerden başlayacak



yeniden başlatılan sorgunun ilk batch'i: 25 | endOffset: {'btc-tx': {'2': 15620, '1': 15446, '0': 15581}}


alarm dosyası: 3182 satır, 3182 tekil txId → tekrar eden kayıt: 0


İki olası durum var: kesinti bir batch'in **ortasına** denk gelirse, yeniden başlatılan sorgu aynı batch numarasını aynı offset aralığıyla tekrar çalıştırır; batch'ler **arasına** denk gelirse sıradaki batch tam olarak son commit edilen offset'ten başlar. Her iki durumda da çıktıda tekrar eden kayıt yok. Kaynağın tekrar oynatılabilmesi ve sink'in idempotent commit'i birlikte exactly-once sağlıyor.

In [13]:
def graceful_stop(q, timeout=15):
    t = time.time()
    while q.status["isTriggerActive"] and time.time() - t < timeout:   # çalışan batch'in bitmesini bekle
        time.sleep(0.2)
    q.stop()

spark.sparkContext.setLogLevel("OFF")
for q in spark.streams.active:
    graceful_stop(q)
if producer_proc:
    producer_proc.terminate()
final = metrics()
print("Toplam işlenen etiketli işlem:", final["TP"] + final["FP"] + final["FN"] + final["TN"],
      "| unknown:", final["unknown_total"], f"(bunların {final['unknown_flagged']} tanesi alarm)")
pd.DataFrame([final])

Toplam işlenen etiketli işlem: 11184 | unknown: 35463 (bunların 2798 tanesi alarm)


,TP,FP,FN,TN,unknown_flagged,unknown_total,batches,precision,recall,F1
0,371,13,265,10535,2798,35463,16,0.966,0.583,0.727


## Değerlendirme

_(Sayılar, 2 çekirdekli bir makinede `--from-step 40 --rate 400` ile yapılan test çalıştırmasından alınmıştır; sizin çalıştırmanızda değişecektir.)_

* **Eğitim ve servis tutarlılığı:** Canlı izlemedeki zaman adımı bazlı F1 değerleri (adım 40: 0.778, 41: 0.96, 42: 0.875, 43: 0.0, 49: 0.035), Aşama 2'deki batch değerlendirmesiyle **birebir aynı**. Aynı `PipelineModel` ve aynı dönüşümler streaming'de de kullanıldığı için bu beklenen sonuç, ve kodun doğru çalıştığını kanıtlıyor.
* **Drift canlı olarak görülüyor:** 43. adımdan itibaren recall çöküyor (kümülatif F1 ~0.92'den ~0.73'e iniyor). Etiketsiz bir sinyal de var: pencere başına alarm oranı 40–42. adımları kapsayan pencerede ~%9.0 iken 43 sonrası pencerelerde ~%5.6–6.3'e düşüyor. Etiketler gecikmeli geldiğinde bile **alarm oranındaki ani değişim** tek başına bir drift alarmı olarak kullanılabilir.
* **Watermark:** Geç damgalanan kayıtların bir kısmı atıldı (`watermark ile atılan`). İnce bir nokta: Akışın ilk saniyelerinde watermark henüz ilerlemediği için o sırada gelen geç kayıtlar kabul edilir. Watermark "en büyük olay zamanı − 1 dk" olarak tanımlandığından, ancak yeterince ileri tarihli olay görüldükten sonra devreye girer.
* **Exactly-once:** Durdurup yeniden başlatma sonrasında alarm dosyasında tekrar eden kayıt sayısı 0.
* **Gecikme ve kapasite:** Sorgu başına yaklaşık 220–300 satır/sn işlendi. Üç sorgu Kafka'yı ve modeli ayrı ayrı çalıştırdığı için toplam iş üç katı.

**Structured Streaming'in DStream'e göre burada kullandığımız avantajları:** (1) Streaming ve batch aynı DataFrame API'sini kullanıyor, böylece Aşama 2'deki model ve dönüşümler değişmeden çalıştı. (2) Olay zamanı pencereleme ve watermark yerleşik. (3) Checkpoint ve tekrar oynatılabilir kaynak (Kafka) ile kayıt kaybı olmadan yeniden başlatma mümkün.

In [14]:
spark.stop()